# CICIDS Baseline IDS Model - Optimized Version

**Optimizations Applied:**
- ✅ Uses preprocessed and scaled data
- ✅ Improved Random Forest hyperparameters
- ✅ 5-fold cross-validation
- ✅ Comprehensive evaluation metrics
- ✅ Feature importance analysis
- ✅ Model versioning with metadata
- ✅ Multiclass classification (15 attack types)

**Dataset:** CICIDS 2017 Network Intrusion Detection
**Classes:** 15 (1 benign + 14 attack types)
**Features:** 45 (after preprocessing and selection)

## 1. Imports and Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import joblib
from datetime import datetime

# Sklearn imports
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, cross_validate, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    ConfusionMatrixDisplay
)

# Visualization settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("✓ All imports successful")

ModuleNotFoundError: No module named 'seaborn'

## 2. Configuration

In [2]:
# Paths
DATA_DIR = Path("../data/processed")
MODEL_DIR = Path("../models/baseline")
PREPROCESSING_DIR = Path("../models/preprocessing")

# Ensure directories exist
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Random seed for reproducibility
RANDOM_STATE = 42

print(f"Data directory: {DATA_DIR}")
print(f"Model directory: {MODEL_DIR}")
print(f"Random state: {RANDOM_STATE}")

NameError: name 'Path' is not defined

## 3. Load Preprocessed Data

In [3]:
# Load training data
print("Loading training data...")
X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze()

# Load validation data
print("Loading validation data...")
X_val = pd.read_csv(DATA_DIR / "X_val.csv")
y_val = pd.read_csv(DATA_DIR / "y_val.csv").squeeze()

# Load test data
print("Loading test data...")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")
y_test = pd.read_csv(DATA_DIR / "y_test.csv").squeeze()

print(f"\n✓ Data loaded successfully")
print(f"Train shape: X={X_train.shape}, y={y_train.shape}")
print(f"Val shape:   X={X_val.shape}, y={y_val.shape}")
print(f"Test shape:  X={X_test.shape}, y={y_test.shape}")

Loading training data...


NameError: name 'DATA_DIR' is not defined

## 4. Load Preprocessing Metadata

In [ ]:
# Load label names
with open(PREPROCESSING_DIR / "label_names.json", 'r') as f:
    label_names = json.load(f)

# Load feature names
with open(PREPROCESSING_DIR / "feature_names.json", 'r') as f:
    feature_names = json.load(f)

print(f"Number of features: {len(feature_names)}")
print(f"Number of classes: {len(label_names)}")
print(f"\nClass labels: {label_names}")

## 5. Exploratory Data Analysis

In [ ]:
# Class distribution in training set
train_dist = y_train.value_counts().sort_index()

plt.figure(figsize=(14, 6))
plt.bar(range(len(train_dist)), train_dist.values, color='steelblue', alpha=0.7)
plt.xlabel('Class ID', fontsize=12)
plt.ylabel('Sample Count', fontsize=12)
plt.title('Training Set - Class Distribution', fontsize=14, fontweight='bold')
plt.xticks(range(len(label_names)), label_names, rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Print distribution
print("\nClass Distribution (Training Set):")
for idx, count in train_dist.items():
    pct = (count / len(y_train)) * 100
    print(f"{idx:2d}. {label_names[idx]:30s}: {count:8,} ({pct:5.2f}%)")

## 6. Train Optimized Random Forest Model

In [ ]:
# Optimized Random Forest hyperparameters
rf_model = RandomForestClassifier(
    n_estimators=500,           # Increased from 200
    max_depth=30,               # Limited to prevent overfitting
    min_samples_split=10,       # Prevent overfitting on small samples
    min_samples_leaf=4,         # Ensure leaf nodes have enough samples
    max_features='sqrt',        # Use sqrt of features at each split
    class_weight='balanced',    # Handle class imbalance
    oob_score=True,            # Out-of-bag score estimation
    bootstrap=True,
    n_jobs=-1,                 # Use all CPU cores
    random_state=RANDOM_STATE,
    verbose=1
)

print("Random Forest Configuration:")
print(f"  - Trees: {rf_model.n_estimators}")
print(f"  - Max depth: {rf_model.max_depth}")
print(f"  - Class weighting: {rf_model.class_weight}")
print(f"  - OOB score: {rf_model.oob_score}")

In [ ]:
# Train the model
print("\nTraining Random Forest model...")
print("This may take several minutes...\n")

training_start = datetime.now()
rf_model.fit(X_train, y_train)
training_time = (datetime.now() - training_start).total_seconds()

print(f"\n✓ Training complete in {training_time:.2f} seconds")
print(f"OOB Score: {rf_model.oob_score_:.4f}")

## 7. Cross-Validation

In [ ]:
# 5-Fold Stratified Cross-Validation
print("Performing 5-fold stratified cross-validation...")
print("This may take 10-15 minutes...\n")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_scores = cross_validate(
    estimator=rf_model,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=['accuracy', 'f1_weighted', 'precision_weighted', 'recall_weighted'],
    return_train_score=True,
    n_jobs=-1,
    verbose=1
)

print("\n" + "="*60)
print("CROSS-VALIDATION RESULTS (5-Fold)")
print("="*60)

for metric_name in ['accuracy', 'f1_weighted', 'precision_weighted', 'recall_weighted']:
    train_scores = cv_scores[f'train_{metric_name}']
    test_scores = cv_scores[f'test_{metric_name}']
    
    print(f"\n{metric_name.replace('_', ' ').title()}:")
    print(f"  Train: {train_scores.mean():.4f} ± {train_scores.std():.4f}")
    print(f"  Val:   {test_scores.mean():.4f} ± {test_scores.std():.4f}")

print("\n" + "="*60)

## 8. Make Predictions

In [ ]:
# Predictions on validation set
print("Making predictions on validation set...")
y_val_pred = rf_model.predict(X_val)
y_val_proba = rf_model.predict_proba(X_val)

# Predictions on test set
print("Making predictions on test set...")
y_test_pred = rf_model.predict(X_test)
y_test_proba = rf_model.predict_proba(X_test)

print("✓ Predictions complete")

## 9. Comprehensive Evaluation

In [ ]:
# Validation Set Metrics
val_accuracy = accuracy_score(y_val, y_val_pred)
val_f1 = f1_score(y_val, y_val_pred, average='weighted')
val_precision = precision_score(y_val, y_val_pred, average='weighted')
val_recall = recall_score(y_val, y_val_pred, average='weighted')

# Test Set Metrics
test_accuracy = accuracy_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred, average='weighted')
test_precision = precision_score(y_test, y_test_pred, average='weighted')
test_recall = recall_score(y_test, y_test_pred, average='weighted')

print("="*60)
print("MODEL PERFORMANCE")
print("="*60)
print(f"\n{'Metric':<20} {'Validation':<15} {'Test':<15}")
print("-"*60)
print(f"{'Accuracy':<20} {val_accuracy:<15.4f} {test_accuracy:<15.4f}")
print(f"{'F1-Score (Weighted)':<20} {val_f1:<15.4f} {test_f1:<15.4f}")
print(f"{'Precision (Weighted)':<20} {val_precision:<15.4f} {test_precision:<15.4f}")
print(f"{'Recall (Weighted)':<20} {val_recall:<15.4f} {test_recall:<15.4f}")
print("="*60)

In [ ]:
# Detailed classification report for test set
print("\nDETAILED CLASSIFICATION REPORT (Test Set)")
print("="*80)
print(classification_report(y_test, y_test_pred, target_names=label_names, digits=4))

## 10. Confusion Matrix

In [ ]:
# Confusion matrix for test set
cm = confusion_matrix(y_test, y_test_pred)

# Plot confusion matrix
plt.figure(figsize=(16, 14))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(cmap='Blues', xticks_rotation=45, values_format='d')
plt.title('Confusion Matrix - Test Set', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Confusion matrix saved to models/baseline/confusion_matrix.png")

## 11. Feature Importance Analysis

In [ ]:
# Get feature importances
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

# Save to CSV
feature_importance_df.to_csv(MODEL_DIR / 'feature_importance.csv', index=False)

# Plot top 20 features
top_n = 20
top_features = feature_importance_df.head(top_n)

plt.figure(figsize=(12, 8))
plt.barh(range(top_n), top_features['importance'].values, color='steelblue', alpha=0.7)
plt.yticks(range(top_n), top_features['feature'].values)
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title(f'Top {top_n} Most Important Features', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 10 Most Important Features:")
for idx, row in feature_importance_df.head(10).iterrows():
    print(f"{row['feature']:40s}: {row['importance']:.6f}")

print("\n✓ Feature importance saved to models/baseline/feature_importance.csv")

## 12. Save Model with Metadata

In [ ]:
# Create comprehensive model metadata
model_metadata = {
    'model_info': {
        'model_id': 'rf_baseline_v2',
        'model_type': 'RandomForestClassifier',
        'created_at': datetime.now().isoformat(),
        'training_time_seconds': training_time,
        'random_state': RANDOM_STATE
    },
    'hyperparameters': rf_model.get_params(),
    'data_info': {
        'num_features': len(feature_names),
        'num_classes': len(label_names),
        'train_samples': len(X_train),
        'val_samples': len(X_val),
        'test_samples': len(X_test)
    },
    'performance': {
        'oob_score': float(rf_model.oob_score_),
        'cv_accuracy_mean': float(cv_scores['test_accuracy'].mean()),
        'cv_accuracy_std': float(cv_scores['test_accuracy'].std()),
        'cv_f1_mean': float(cv_scores['test_f1_weighted'].mean()),
        'cv_f1_std': float(cv_scores['test_f1_weighted'].std()),
        'val_accuracy': float(val_accuracy),
        'val_f1_score': float(val_f1),
        'val_precision': float(val_precision),
        'val_recall': float(val_recall),
        'test_accuracy': float(test_accuracy),
        'test_f1_score': float(test_f1),
        'test_precision': float(test_precision),
        'test_recall': float(test_recall)
    },
    'feature_names': feature_names,
    'label_names': label_names
}

# Save model
model_path = MODEL_DIR / 'random_forest_v2.joblib'
joblib.dump(rf_model, model_path, compress=3)
print(f"✓ Model saved to {model_path}")

# Save metadata
metadata_path = MODEL_DIR / 'model_metadata_v2.json'
with open(metadata_path, 'w') as f:
    json.dump(model_metadata, f, indent=2)
print(f"✓ Metadata saved to {metadata_path}")

# Save model package (model + all artifacts)
model_package = {
    'model': rf_model,
    'feature_names': feature_names,
    'label_names': label_names,
    'metadata': model_metadata
}

package_path = MODEL_DIR / 'model_package_v2.joblib'
joblib.dump(model_package, package_path, compress=3)
print(f"✓ Complete model package saved to {package_path}")

## 13. Summary

In [ ]:
print("\n" + "="*80)
print("BASELINE MODEL TRAINING COMPLETE")
print("="*80)

print("\n📊 Model Configuration:")
print(f"  - Algorithm: Random Forest")
print(f"  - Trees: {rf_model.n_estimators}")
print(f"  - Max Depth: {rf_model.max_depth}")
print(f"  - Training Time: {training_time:.2f}s")

print("\n📈 Performance (Test Set):")
print(f"  - Accuracy:  {test_accuracy:.4f}")
print(f"  - F1-Score:  {test_f1:.4f}")
print(f"  - Precision: {test_precision:.4f}")
print(f"  - Recall:    {test_recall:.4f}")

print("\n💾 Saved Artifacts:")
print(f"  - Model: {model_path}")
print(f"  - Metadata: {metadata_path}")
print(f"  - Package: {package_path}")
print(f"  - Confusion Matrix: {MODEL_DIR / 'confusion_matrix.png'}")
print(f"  - Feature Importance: {MODEL_DIR / 'feature_importance.csv'}")

print("\n" + "="*80)
print("✅ Baseline model successfully trained and saved!")
print("="*80)